# PHYS 132 Activity 3: Polytropes and the Sun

In the lecture we reduced the structure of a polytrope, a star with $P = K\rho^{1+1/n}$ throughout, to the
Lane--Emden equation,
$$\frac{1}{\xi^2}\frac{d}{d\xi}\left(\xi^2\frac{d\theta}{d\xi}\right) = -\theta^n , \qquad \theta(0) = 1, \quad \theta'(0) = 0 ,$$
with $\rho = \rho_c\theta^n$ and $r = a\xi$. In this activity you will solve it, and then ask how well the Sun is
described by a polytrope, using the standard solar model of Activities 1 and 2.

Run the first cell as it is. It loads the solar model into arrays `m`, `r`, `T`, `rho`, `P`, `L`, `X` and `Y`, one
value per shell, in cgs units, defines the constants, and imports `solve_ivp` for integrating differential
equations.

Each part says what to find but not how. Work out how to do it before writing any code, and open the hint only if
you are stuck.

In [ ]:
# Run this cell first. It loads the solar model and defines the constants we need, in cgs units.
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid, trapezoid, solve_ivp
np.seterr(divide="ignore", invalid="ignore")   # the innermost shells have L = 0 and repeated values

URL = "https://www.sns.ias.edu/~jnb/SNdata/Export/BS2005/bs05op.dat"
if not os.path.exists("bs05op.dat"):
    # If this download fails, get bs05op.dat from Canvas and upload it next to this notebook.
    urllib.request.urlretrieve(URL, "bs05op.dat")

rows = []
for line in open("bs05op.dat"):
    words = line.split()
    if len(words) == 12 and words[0][0].isdigit():
        rows.append([float(w) for w in words])
table = np.array(rows)

G    = 6.674e-8      # gravitational constant, cm^3 g^-1 s^-2
a    = 7.566e-15     # radiation constant, erg cm^-3 K^-4
c    = 2.998e10      # speed of light, cm s^-1
k_B  = 1.381e-16     # Boltzmann constant, erg K^-1
m_u  = 1.661e-24     # atomic mass unit, g
Msun = 1.989e33      # g
Rsun = 6.9598e10     # cm     (the solar radius and luminosity used to build the model)
Lsun = 3.8418e33     # erg s^-1

m   = table[:, 0] * Msun    # mass inside the shell, g
r   = table[:, 1] * Rsun    # radius of the shell, cm
T   = table[:, 2]           # temperature, K
rho = table[:, 3]           # density, g cm^-3
P   = table[:, 4]           # pressure, dyn cm^-2
L   = table[:, 5] * Lsun    # luminosity flowing out through the shell, erg s^-1
X   = table[:, 6]           # hydrogen mass fraction
Y   = table[:, 7]           # helium-4 mass fraction

print(f"{len(r)} shells, from r = {r[0]/Rsun:.4f} R_sun to r = {r[-1]/Rsun:.3f} R_sun")

## 1. Solving the Lane--Emden equation

Write a function that integrates the Lane--Emden equation for a given $n$ from the centre out to the surface, the
first zero of $\theta$, and returns $\xi_1$, $\omega_n = -\xi_1^2\theta'(\xi_1)$ and the ratio of the central to the
mean density, $\rho_c/\bar\rho$. Check it against the exact solution for $n = 1$, $\theta = \sin\xi/\xi$, and then
find the three numbers for $n = 3/2$ and $n = 3$.

<details><summary><b>Hint</b></summary>

Write the equation as two first-order equations for $\theta$ and $\theta'$; `solve_ivp` integrates such a system.
The term $2\theta'/\xi$ cannot be evaluated at $\xi = 0$, so start at a small $\xi_0$, such as $10^{-6}$, with the
series $\theta \approx 1 - \xi^2/6$ and $\theta' \approx -\xi/3$. The `events` option of `solve_ivp` stops the
integration where a function of the solution, here $\theta$, crosses zero. Use `max(theta, 0)**n`, since a
fractional power of a negative number is not defined. For the mean density, use $M = 4\pi a^3\rho_c\,\omega_n$ and
$R = a\,\xi_1$.
</details>

In [ ]:
# Part 1


Why does the integration have to start slightly away from the centre, and how do you know that starting at
$\xi_0$ rather than at 0 does not affect your answers?

## 2. How concentrated is the Sun?

Compute $\rho_c/\bar\rho$ for polytropes from $n = 0$ to $n = 4.5$ and plot it against $n$. Find the Sun's own value
from the solar model, taking its mass and radius to be $M_\odot$ and $R_\odot$, and find the index of the polytrope
that is as centrally concentrated as the Sun.

<details><summary><b>Hint</b></summary>

`np.arange(0, 4.75, 0.25)` gives the values of $n$. The ratio grows by orders of magnitude, so use a logarithmic
axis, and interpolate in $\log(\rho_c/\bar\rho)$ to find the Sun's $n$.
</details>

In [ ]:
# Part 2


What happens to $\rho_c/\bar\rho$ as $n$ approaches 5, and why?

## 3. Is the Sun a polytrope?

A polytrope has the same exponent $\Gamma = d\ln P/d\ln\rho = 1 + 1/n$ throughout. Measure $\Gamma$ through the
solar model, from near the centre out to $0.96\,R_\odot$, and plot it against $r/R_\odot$ with reference lines at
$5/3$ ($n = 3/2$), $4/3$ ($n = 3$) and 1 ($n \to \infty$).

<details><summary><b>Hint</b></summary>

The innermost shells of the table repeat the same values, so differencing neighbouring shells divides zero by zero.
Interpolate $\ln P$ and $\ln\rho$ onto evenly spaced radii, every $0.01\,R_\odot$, and use `np.gradient(y, x)`,
which accepts unevenly spaced `x`.
</details>

In [ ]:
# Part 3


1. In the lecture we said that a layer in which convection keeps the entropy uniform has $P \propto \rho^{5/3}$.
   Where in the Sun is $\Gamma$ close to $5/3$, and how does that compare with the radius you found in Activity 1,
   Part 3, beyond which radiation alone could not carry the luminosity?
2. Over what range does $\Gamma$ run in the radiative interior, and is $n = 3$ a fair description there?
3. Near the centre $\Gamma$ falls to 1, the value for an isothermal gas, and below it, although the temperature
   there is far from uniform. Since $P/\rho = k_BT/\bar m$, what else must be changing? Look at $X$ in the core.